# Expectation

**Concept 25** of the math-foundations learning map.

The expectation $E[X]$ is the probability-weighted average of a random variable's values. Discretely $E[X] = \sum_k k\, p_X(k)$; continuously $E[X] = \int x f_X(x)\,dx$; in general $E[X] = \int_\Omega X\,dP$.

This notebook demonstrates: (1) computing $E[X]$ for a Binomial, (2) linearity of expectation, (3) Jensen's inequality.

In [ ]:
import math
import random
random.seed(0)

def binomial_pmf(n, p, k):
    return math.comb(n, k) * (p**k) * ((1-p)**(n-k))

def sample_binomial(n, p):
    return sum(1 for _ in range(n) if random.random() < p)

def mean(xs):
    return sum(xs) / len(xs)


## 1. $E[X]$ for $X \sim \text{Binomial}(10, 0.4)$

Analytic: $E[X] = np = 4.0$. We verify via the PMF sum and Monte Carlo.

In [ ]:
n, p = 10, 0.4
analytic = sum(k * binomial_pmf(n, p, k) for k in range(n+1))
N = 10000
samples = [sample_binomial(n, p) for _ in range(N)]
empirical = mean(samples)
print(f'analytic  E[X] = {analytic:.6f}')
print(f'np        E[X] = {n*p:.6f}')
print(f'empirical (N={N}) = {empirical:.6f}')


## 2. Linearity: $E[X + Y] = E[X] + E[Y]$

Take independent $X \sim \text{Bin}(10, 0.4)$ and $Y \sim \text{Bern}(0.7)$. Linearity holds regardless of independence.

In [ ]:
pY = 0.7
x = [sample_binomial(n, p) for _ in range(N)]
y = [1 if random.random() < pY else 0 for _ in range(N)]
EX, EY = mean(x), mean(y)
EXY = mean([a+b for a,b in zip(x, y)])
print(f'E[X]        = {EX:.6f}')
print(f'E[Y]        = {EY:.6f}')
print(f'E[X]+E[Y]   = {EX+EY:.6f}')
print(f'E[X+Y]      = {EXY:.6f}')
assert math.isclose(EX+EY, EXY, abs_tol=1e-9)


## 3. Jensen's inequality: $E[X^2] \ge (E[X])^2$

Since $\phi(x) = x^2$ is convex, Jensen gives $\phi(E[X]) \le E[\phi(X)]$. The gap is precisely $\operatorname{Var}(X)$.

In [ ]:
EX2 = mean([v*v for v in x])
print(f'E[X^2]   = {EX2:.6f}')
print(f'(E[X])^2 = {EX**2:.6f}')
print(f'Var(X)   = {EX2 - EX**2:.6f}  (>= 0)')
assert EX2 >= EX**2 - 1e-9


## Takeaways

- Every loss in ML is an expectation: flow-matching's $E_{x,z_0,t}[\|v - (z_1-z_0)\|^2]$, GRPO's centred reward, ELBO, policy gradients.
- **Linearity** lets us compute $E[\text{Bin}(n,p)] = np$ without combinatorial sums.
- **Jensen** is the engine behind ELBO bounds and KL non-negativity.